In [ ]:
import torch
import numpy as np

# Load the model checkpoint
model = torch.load("LCZNet.h5")
model.eval()


In [ ]:
import rasterio

# Load Sentinel-2 image (assuming a GeoTIFF with 10m resolution)
image_path = "sentinel2_image.tif"
with rasterio.open(image_path) as src:
    sentinel_data = src.read()

# Normalize data if needed
sentinel_data = sentinel_data.astype(np.float32) / 10000.0


In [ ]:
import torch

# Sentinel-2 bands expected: [B2, B3, B4, B8, ...]
# Reshape and convert to tensor
input_tensor = torch.from_numpy(sentinel_data).unsqueeze(0)


In [ ]:
# Inference
with torch.no_grad():
    output = model(input_tensor)

# Get predicted LCZ classes
lcz_map = torch.argmax(output, dim=1).squeeze().numpy()


In [ ]:
from rasterio.transform import from_origin

# Save the LCZ map
output_file = "lcz_map.tif"
transform = src.transform

# Save as GeoTIFF
with rasterio.open(
    output_file, "w",
    driver="GTiff",
    height=lcz_map.shape[0],
    width=lcz_map.shape[1],
    count=1,
    dtype=lcz_map.dtype,
    crs=src.crs,
    transform=transform
) as dst:
    dst.write(lcz_map, 1)

print(f"LCZ map saved at {output_file}")
